In [0]:
dbutils.widgets.text("fecha_proceso", "2026-07-24", "Fecha de Proceso")
fecha_proceso = dbutils.widgets.get("fecha_proceso")
print(f"Procesando datos para la fecha: {fecha_proceso}")

# Verificación de acceso a los containers
print("Verificando acceso a containers...")
display(dbutils.fs.ls("abfss://raw@sttptransandino01.dfs.core.windows.net/"))

Procesando datos para la fecha: 2026-07-24
Verificando acceso a containers...


path,name,size,modificationTime
abfss://raw@sttptransandino01.dfs.core.windows.net/clientes_logistica.csv,clientes_logistica.csv,3800,1784857887000
abfss://raw@sttptransandino01.dfs.core.windows.net/envios.csv,envios.csv,56032,1784857887000
abfss://raw@sttptransandino01.dfs.core.windows.net/incidencias_2023.csv,incidencias_2023.csv,9740,1784857887000
abfss://raw@sttptransandino01.dfs.core.windows.net/incidencias_2024.csv,incidencias_2024.csv,6415,1784857887000
abfss://raw@sttptransandino01.dfs.core.windows.net/rutas.csv,rutas.csv,1312,1784857887000
abfss://raw@sttptransandino01.dfs.core.windows.net/transportistas.csv,transportistas.csv,1232,1784857887000


In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, lit

# Definición de schemas explícitos (todo String, según Consigna 4.2)
# Orden corregido según el CSV real: id_envio, id_cliente, id_transportista, id_ruta, peso_kg, monto_flete, fecha_envio, estado, updated_at
schema_envios = StructType([
    StructField("id_envio", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("id_transportista", StringType(), True),
    StructField("id_ruta", StringType(), True),
    StructField("peso_kg", StringType(), True),
    StructField("monto_flete", StringType(), True),
    StructField("fecha_envio", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("updated_at", StringType(), True),
])

base_path = f"abfss://landing@sttptransandino01.dfs.core.windows.net"

def ingest_bronze(schema, nombre_archivo, nombre_tabla):
    path = f"{base_path}/{nombre_archivo.replace('.csv','')}/{fecha_proceso}/{nombre_archivo}"
    df = (spark.read
          .schema(schema)
          .option("header", "true")
          .csv(path))
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("fecha_proceso", lit(fecha_proceso))
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"tp_transandino.bronze.{nombre_tabla}")
    print(f"✅ {nombre_tabla}: {df.count()} filas cargadas")
    return df

df_bronze_envios = ingest_bronze(schema_envios, "envios.csv", "envios")

✅ envios: 600 filas cargadas


In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, lit

schema_transportistas = StructType([
    StructField("id_transportista", StringType(), True),
    StructField("nombre_transportista", StringType(), True),
    StructField("tipo_vehiculo", StringType(), True),
    StructField("capacidad_kg", StringType(), True),
    StructField("zona_cobertura", StringType(), True),
])

schema_rutas = StructType([
    StructField("id_ruta", StringType(), True),
    StructField("origen", StringType(), True),
    StructField("destino", StringType(), True),
    StructField("distancia_km", StringType(), True),
    StructField("tipo_ruta", StringType(), True),
])

schema_clientes = StructType([
    StructField("id_cliente", StringType(), True),
    StructField("nombre_cliente", StringType(), True),
    StructField("zona", StringType(), True),
    StructField("tipo_cliente", StringType(), True),
])

schema_incidencias = StructType([
    StructField("id_incidencia", StringType(), True),
    StructField("id_envio", StringType(), True),
    StructField("tipo_incidencia", StringType(), True),
    StructField("estado_resolucion", StringType(), True),
    StructField("resolucion", StringType(), True),
])


df_bronze_transportistas = ingest_bronze(schema_transportistas, "transportistas.csv", "transportistas")

df_bronze_rutas = ingest_bronze(schema_rutas, "rutas.csv", "rutas")
df_bronze_clientes = ingest_bronze(schema_clientes, "clientes_logistica.csv", "clientes_logistica")
df_bronze_incidencias_2023 = ingest_bronze(schema_incidencias, "incidencias_2023.csv", "incidencias_2023_temp")

✅ transportistas: 20 filas cargadas
✅ rutas: 30 filas cargadas
✅ clientes_logistica: 60 filas cargadas
✅ incidencias_2023_temp: 120 filas cargadas


In [0]:
df_bronze_transportistas.count(), df_bronze_rutas.count(), df_bronze_clientes.count(), df_bronze_incidencias_2023.count()

(20, 30, 60, 120)

In [0]:
df_bronze_incidencias_2023.printSchema()
df_bronze_incidencias_2023.show(5)

root
 |-- id_incidencia: string (nullable = true)
 |-- id_envio: string (nullable = true)
 |-- tipo_incidencia: string (nullable = true)
 |-- estado_resolucion: string (nullable = true)
 |-- resolucion: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- fecha_proceso: string (nullable = false)

+-------------+---------+-------------------+-----------------+--------------------+--------------------+-------------+
|id_incidencia| id_envio|    tipo_incidencia|estado_resolucion|          resolucion| ingestion_timestamp|fecha_proceso|
+-------------+---------+-------------------+-----------------+--------------------+--------------------+-------------+
|    INC-00001|ENV-00092|     Accidente vial|       2023-08-30|Reentrega programada|2026-07-24 23:09:...|   2026-07-24|
|    INC-00002|ENV-00226|       Robo parcial|       2023-12-26|Indemnización apr...|2026-07-24 23:09:...|   2026-07-24|
|    INC-00003|ENV-00036|       Robo parcial|       2023-03-30|Reentre

In [0]:
df_bronze_incidencias_2024 = ingest_bronze(schema_incidencias, "incidencias_2024.csv", "incidencias_2024_temp")

✅ incidencias_2024_temp: 80 filas cargadas


In [0]:
print("=== Resumen Bronze ===")
for tabla in ["envios", "transportistas", "rutas", "clientes_logistica", "incidencias_2023_temp", "incidencias_2024_temp"]:
    count = spark.table(f"tp_transandino.bronze.{tabla}").count()
    print(f"{tabla}: {count} filas")

=== Resumen Bronze ===
envios: 600 filas
transportistas: 20 filas
rutas: 30 filas
clientes_logistica: 60 filas
incidencias_2023_temp: 120 filas
incidencias_2024_temp: 80 filas


In [0]:
df_bronze_envios.select("fecha_envio").distinct().show(30, truncate=False)

+-------------------+
|fecha_envio        |
+-------------------+
|2024/06/23 00:00:00|
|29/02/2024         |
|2024/12/06 00:00:00|
|09-20-2023         |
|2023/08/04 00:00:00|
|26/12/2024         |
|2024/04/25 00:00:00|
|12-25-2024         |
|2023-06-19         |
|2023/02/06         |
|2022/09/17 00:00:00|
|01-14-2024         |
|2022-11-30         |
|2023/01/13 00:00:00|
|2024-10-17         |
|01-10-2024         |
|2024/12/27         |
|13/12/2023         |
|27/04/2023         |
|2024/11/28 00:00:00|
|2024-08-13         |
|02/04/2024         |
|2024-01-19         |
|2024/08/30         |
|10-07-2023         |
|26/06/2022         |
|06-25-2022         |
|2024/10/20 00:00:00|
|15/11/2024         |
|07-19-2024         |
+-------------------+
only showing top 30 rows


In [0]:
df_check = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{base_path}/envios/{fecha_proceso}/envios.csv")
df_check.printSchema()
df_check.show(5)

root
 |-- id_envio: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_transportista: string (nullable = true)
 |-- id_ruta: string (nullable = true)
 |-- peso_kg: string (nullable = true)
 |-- monto_flete: double (nullable = true)
 |-- fecha_envio: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+---------+----------+----------------+-------+-------+-----------+-----------+-----------+-------------------+
| id_envio|id_cliente|id_transportista|id_ruta|peso_kg|monto_flete|fecha_envio|     estado|         updated_at|
+---------+----------+----------------+-------+-------+-----------+-----------+-----------+-------------------+
|ENV-00001|   CLI-051|         TRA-016|RUT-024| 457.05|    6475.38| 2023/01/06| entregado |2023-10-02 00:00:00|
|ENV-00002|   CLI-044|         TRA-004|RUT-012| 373.89|     560.05| 2022-09-08|  cancelado|2024-06-17 00:00:00|
|ENV-00003|   CLI-012|         TRA-009|RUT-008| 429.26|  

In [0]:
from pyspark.sql.functions import (
    col, when, trim, lower, coalesce, expr, split
)

# Paso 1: separar la parte de fecha (sin hora) de fecha_envio
df_silver_envios = df_bronze_envios.withColumn(
    "fecha_solo", split(col("fecha_envio"), " ")[0]
)

# Paso 2: parseo con coalesce usando try_to_date (tolera formatos inválidos -> null)
df_silver_envios = df_silver_envios.withColumn(
    "fecha_envio_parsed",
    coalesce(
        expr("try_to_date(fecha_solo, 'yyyy/MM/dd')"),
        expr("try_to_date(fecha_solo, 'dd/MM/yyyy')"),
        expr("try_to_date(fecha_solo, 'yyyy-MM-dd')"),
        expr("try_to_date(fecha_solo, 'MM-dd-yyyy')")
    )
)

# Paso 3: limpieza de tipos numéricos (N/A -> null)
df_silver_envios = df_silver_envios.withColumn(
    "peso_kg",
    when(col("peso_kg").isin("N/A", "n/a", ""), None).otherwise(col("peso_kg").cast("double"))
).withColumn(
    "monto_flete",
    when(col("monto_flete").isin("N/A", "n/a", ""), None).otherwise(col("monto_flete").cast("double"))
)

# Paso 4: normalización de estado (minúsculas, sin espacios extra)
df_silver_envios = df_silver_envios.withColumn(
    "estado", trim(lower(col("estado")))
)

# Paso 5: quitar duplicados por id_envio (quedándonos con el registro más reciente por updated_at)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("id_envio").orderBy(col("updated_at").desc())
df_silver_envios = (df_silver_envios
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn", "fecha_solo", "fecha_envio")
    .withColumnRenamed("fecha_envio_parsed", "fecha_envio")
)

# Verificación
print(f"Filas antes de dedup: {df_bronze_envios.count()}")
print(f"Filas después de dedup: {df_silver_envios.count()}")
print(f"Fechas nulas tras parseo: {df_silver_envios.filter(col('fecha_envio').isNull()).count()}")

df_silver_envios.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.silver.envios")
display(df_silver_envios.limit(10))

Filas antes de dedup: 600
Filas después de dedup: 600
Fechas nulas tras parseo: 0


id_envio,id_cliente,id_transportista,id_ruta,peso_kg,monto_flete,estado,updated_at,ingestion_timestamp,fecha_proceso,fecha_envio
ENV-00001,CLI-051,TRA-016,RUT-024,457.05,6475.38,entregado,2023-10-02 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2023-01-06
ENV-00002,CLI-044,TRA-004,RUT-012,373.89,560.05,cancelado,2024-06-17 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2022-09-08
ENV-00003,CLI-012,TRA-009,RUT-008,429.26,3744.01,entregado,2022-10-29 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2024-08-28
ENV-00004,CLI-001,TRA-003,RUT-018,521.89,7416.83,entregado,2023-12-20 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2024-09-05
ENV-00005,CLI-001,TRA-013,RUT-007,722.0,2180.41,cancelado,2022-11-23 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2023-07-22
ENV-00006,CLI-010,TRA-007,RUT-024,444.84,2429.93,entregado,2024-06-13 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2023-03-20
ENV-00007,CLI-059,TRA-017,RUT-024,40.59,1326.87,pendiente,2023-10-11 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2024-06-23
ENV-00008,CLI-057,TRA-019,RUT-020,689.52,2030.03,pendiente,2023-07-21 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2024-10-19
ENV-00009,CLI-043,TRA-011,RUT-022,418.46,1970.39,en_transito,2023-02-27 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2024-02-25
ENV-00010,CLI-030,TRA-020,RUT-015,601.62,7987.76,entregado,2022-08-15 00:00:00,2026-07-24T23:11:33.790Z,2026-07-24,2024-04-18


In [0]:
# Schemas corregidos según columnas reales
schema_transportistas = StructType([
    StructField("id_transportista", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("zona", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("modalidad", StringType(), True),
])

schema_rutas = StructType([
    StructField("id_ruta", StringType(), True),
    StructField("origen", StringType(), True),
    StructField("destino", StringType(), True),
    StructField("tipo_ruta", StringType(), True),
    StructField("distancia_km", StringType(), True),
])

schema_clientes = StructType([
    StructField("id_cliente", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("segmento", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("fecha_alta", StringType(), True),
])


schema_incidencias = StructType([
    StructField("id_incidencia", StringType(), True),
    StructField("id_envio", StringType(), True),
    StructField("tipo_incidencia", StringType(), True),
    StructField("fecha_incidencia", StringType(), True),
    StructField("resolucion", StringType(), True),
    StructField("estado_resolucion", StringType(), True),
])

df_bronze_incidencias_2023 = ingest_bronze(schema_incidencias, "incidencias_2023.csv", "incidencias_2023_temp")
df_bronze_incidencias_2024 = ingest_bronze(schema_incidencias, "incidencias_2024.csv", "incidencias_2024_temp")
df_bronze_transportistas = ingest_bronze(schema_transportistas, "transportistas.csv", "transportistas")
df_bronze_rutas = ingest_bronze(schema_rutas, "rutas.csv", "rutas")
df_bronze_clientes = ingest_bronze(schema_clientes, "clientes_logistica.csv", "clientes_logistica")

✅ incidencias_2023_temp: 120 filas cargadas
✅ incidencias_2024_temp: 80 filas cargadas
✅ transportistas: 20 filas cargadas
✅ rutas: 30 filas cargadas
✅ clientes_logistica: 60 filas cargadas


In [0]:
from pyspark.sql.functions import col, trim, lower, expr

# Silver: transportistas
df_silver_transportistas = (df_bronze_transportistas
    .withColumn("nombre", trim(col("nombre")))
    .withColumn("zona", trim(lower(col("zona"))))
    .withColumn("ciudad", trim(col("ciudad")))
    .withColumn("modalidad", trim(lower(col("modalidad"))))
    .dropDuplicates(["id_transportista"])
)
df_silver_transportistas.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.silver.transportistas")
print(f"✅ silver.transportistas: {df_silver_transportistas.count()} filas")

# Silver: rutas
df_silver_rutas = (df_bronze_rutas
    .withColumn("distancia_km", col("distancia_km").cast("double"))
    .withColumn("origen", trim(col("origen")))
    .withColumn("destino", trim(col("destino")))
    .withColumn("tipo_ruta", trim(lower(col("tipo_ruta"))))
    .dropDuplicates(["id_ruta"])
)
df_silver_rutas.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.silver.rutas")
print(f"✅ silver.rutas: {df_silver_rutas.count()} filas")

# Silver: clientes_logistica
df_silver_clientes = (df_bronze_clientes
    .withColumn("nombre", trim(col("nombre")))
    .withColumn("segmento", trim(lower(col("segmento"))))
    .withColumn("ciudad", trim(col("ciudad")))
    .withColumn("fecha_alta", expr("try_to_date(fecha_alta, 'yyyy-MM-dd')"))
    .dropDuplicates(["id_cliente"])
)
df_silver_clientes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.silver.clientes_logistica")
print(f"✅ silver.clientes_logistica: {df_silver_clientes.count()} filas")


from pyspark.sql.functions import col, trim, lower, expr

df_silver_incidencias_2023 = (df_bronze_incidencias_2023
    .withColumn("fecha_incidencia", expr("try_to_date(fecha_incidencia, 'yyyy-MM-dd')"))
    .withColumn("tipo_incidencia", trim(col("tipo_incidencia")))
    .withColumn("estado_resolucion", trim(lower(col("estado_resolucion"))))
    .withColumn("resolucion", trim(col("resolucion")))
    .dropDuplicates(["id_incidencia"])
)

df_silver_incidencias_2023.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.silver.incidencias")
print(f"✅ silver.incidencias (base 2023): {df_silver_incidencias_2023.count()} filas")

✅ silver.transportistas: 20 filas
✅ silver.rutas: 30 filas
✅ silver.clientes_logistica: 60 filas
✅ silver.incidencias (base 2023): 120 filas


In [0]:
for archivo in ["incidencias_2023", "incidencias_2024"]:
    print(f"=== {archivo} ===")
    df_check = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{base_path}/{archivo}/{fecha_proceso}/{archivo}.csv")
    df_check.printSchema()
    df_check.show(3)
    print()

=== incidencias_2023 ===
root
 |-- id_incidencia: string (nullable = true)
 |-- id_envio: string (nullable = true)
 |-- tipo_incidencia: string (nullable = true)
 |-- fecha_incidencia: date (nullable = true)
 |-- resolucion: string (nullable = true)
 |-- estado_resolucion: string (nullable = true)

+-------------+---------+---------------+----------------+--------------------+-----------------+
|id_incidencia| id_envio|tipo_incidencia|fecha_incidencia|          resolucion|estado_resolucion|
+-------------+---------+---------------+----------------+--------------------+-----------------+
|    INC-00001|ENV-00092| Accidente vial|      2023-08-30|Reentrega programada|         Resuelto|
|    INC-00002|ENV-00226|   Robo parcial|      2023-12-26|Indemnización apr...|         Resuelto|
|    INC-00003|ENV-00036|   Robo parcial|      2023-03-30|Reentrega programada|        Pendiente|
+-------------+---------+---------------+----------------+--------------------+-----------------+
only showing t

### Consigna 4.4 — MERGE INTO incremental de incidencias_2024

In [0]:
from pyspark.sql.functions import col, trim, lower, expr
from delta.tables import DeltaTable

# Limpiar incidencias_2024 igual que 2023
df_silver_incidencias_2024 = (df_bronze_incidencias_2024
    .withColumn("fecha_incidencia", expr("try_to_date(fecha_incidencia, 'yyyy-MM-dd')"))
    .withColumn("tipo_incidencia", trim(col("tipo_incidencia")))
    .withColumn("estado_resolucion", trim(lower(col("estado_resolucion"))))
    .withColumn("resolucion", trim(col("resolucion")))
    .dropDuplicates(["id_incidencia"])
)

print(f"Filas nuevas a mergear (2024): {df_silver_incidencias_2024.count()}")

# Verificación previa: cuántos IDs ya existen (deberían actualizarse) vs nuevos (deberían insertarse)
ids_existentes = spark.table("tp_transandino.silver.incidencias").select("id_incidencia")
existentes = df_silver_incidencias_2024.join(ids_existentes, "id_incidencia", "inner").count()
nuevos = df_silver_incidencias_2024.count() - existentes
print(f"IDs que ya existen (se actualizarán): {existentes}")
print(f"IDs nuevos (se insertarán): {nuevos}")

# MERGE INTO
tabla_delta = DeltaTable.forName(spark, "tp_transandino.silver.incidencias")

(tabla_delta.alias("t")
    .merge(
        df_silver_incidencias_2024.alias("s"),
        "t.id_incidencia = s.id_incidencia"
    )
    .whenMatchedUpdate(set={
        "estado_resolucion": "s.estado_resolucion",
        "resolucion": "s.resolucion"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

count_final = spark.table("tp_transandino.silver.incidencias").count()
print(f"✅ Total filas en silver.incidencias tras MERGE: {count_final}")

Filas nuevas a mergear (2024): 80
IDs que ya existen (se actualizarán): 30
IDs nuevos (se insertarán): 50
✅ Total filas en silver.incidencias tras MERGE: 170


### Consigna 4.5 — Capa Gold (las 2 tablas de KPIs)

Tabla 1: gold_envios_zona

In [0]:
from pyspark.sql.functions import year, month, sum as _sum, avg, count

df_silver_envios = spark.table("tp_transandino.silver.envios")
df_silver_transportistas = spark.table("tp_transandino.silver.transportistas")

df_gold_envios_zona = (df_silver_envios
    .join(df_silver_transportistas, "id_transportista", "inner")
    .withColumn("anio", year(col("fecha_envio")))
    .withColumn("mes", month(col("fecha_envio")))
    .groupBy("zona", "anio", "mes")
    .agg(
        count("id_envio").alias("total_envios"),
        _sum("monto_flete").alias("monto_total"),
        _sum("peso_kg").alias("peso_total"),
        avg("monto_flete").alias("ticket_promedio")
    )
)

df_gold_envios_zona.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.gold.envios_zona")
print(f"✅ gold.envios_zona: {df_gold_envios_zona.count()} filas")
display(df_gold_envios_zona.limit(10))

✅ gold.envios_zona: 151 filas


zona,anio,mes,total_envios,monto_total,peso_total,ticket_promedio
oeste,2022,9,4,16400.11,1968.86,4100.0275
oeste,2024,8,4,16578.98,2048.71,4144.745
centro,2023,3,6,27870.92,3152.33,4645.153333333333
centro,2024,6,6,24198.72,2179.23,4033.1200000000003
centro,2023,12,6,27477.4,2144.95,4579.566666666667
norte,2022,7,4,18675.309999999998,2078.59,4668.827499999999
centro,2023,5,2,7280.639999999999,1316.35,3640.3199999999997
norte,2022,12,1,7324.75,180.66,7324.75
norte,2023,10,5,8592.48,2005.3300000000002,1718.4959999999999
centro,2023,6,4,12079.1,1455.87,3019.775


Tabla 2: gold_envios_tipo_ruta

In [0]:
df_silver_rutas = spark.table("tp_transandino.silver.rutas")

df_gold_envios_tipo_ruta = (df_silver_envios
    .join(df_silver_rutas, "id_ruta", "inner")
    .withColumn("anio", year(col("fecha_envio")))
    .withColumn("mes", month(col("fecha_envio")))
    .groupBy("tipo_ruta", "anio", "mes")
    .agg(
        count("id_envio").alias("total_envios"),
        _sum("monto_flete").alias("monto_total"),
        avg("monto_flete").alias("ticket_promedio"),
        avg("distancia_km").alias("distancia_promedio")
    )
)

df_gold_envios_tipo_ruta.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tp_transandino.gold.envios_tipo_ruta")
print(f"✅ gold.envios_tipo_ruta: {df_gold_envios_tipo_ruta.count()} filas")
display(df_gold_envios_tipo_ruta.limit(10))

✅ gold.envios_tipo_ruta: 93 filas


tipo_ruta,anio,mes,total_envios,monto_total,ticket_promedio,distancia_promedio
aéreo,2024,9,4,23479.97,5869.9925,5200.0
aéreo,2024,12,10,45365.229999999996,4536.522999999999,4702.4
marítimo,2024,10,2,10037.34,5018.67,2695.0
terrestre,2024,5,9,27991.64,3110.182222222222,1096.5555555555557
terrestre,2022,12,7,21391.04,3055.862857142857,1292.7142857142858
marítimo,2024,9,3,14761.949999999999,4920.65,5586.0
marítimo,2022,11,4,9735.019999999999,2433.7549999999997,5050.5
marítimo,2023,5,2,6729.97,3364.985,3129.0
aéreo,2023,9,5,21665.230000000003,4333.046,6903.2
aéreo,2022,6,3,9774.350000000002,3258.1166666666672,5631.666666666667


Verificación de consistencia

In [0]:
total_envios_silver = df_silver_envios.count()
total_envios_gold_zona = df_gold_envios_zona.agg(_sum("total_envios")).collect()[0][0]

diferencia = abs(total_envios_silver - total_envios_gold_zona)
print(f"Total filas en silver.envios: {total_envios_silver}")
print(f"Suma de total_envios en gold.envios_zona: {total_envios_gold_zona}")
print(f"Diferencia: {diferencia}")
print(f"¿Consistente (tolerancia 10)?: {'✅ SI' if diferencia <= 10 else '❌ NO'}")

Total filas en silver.envios: 600
Suma de total_envios en gold.envios_zona: 600
Diferencia: 0
¿Consistente (tolerancia 10)?: ✅ SI


### Consigna 4.6 — Celda de limpieza

In [0]:
# ==========================================
# CELDA DE LIMPIEZA - Descomentar solo si se necesita reiniciar el TP desde cero
# ==========================================

# spark.sql("DROP TABLE IF EXISTS tp_transandino.bronze.envios")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.bronze.transportistas")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.bronze.rutas")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.bronze.clientes_logistica")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.bronze.incidencias_2023_temp")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.bronze.incidencias_2024_temp")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.silver.envios")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.silver.transportistas")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.silver.rutas")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.silver.clientes_logistica")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.silver.incidencias")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.gold.envios_zona")
# spark.sql("DROP TABLE IF EXISTS tp_transandino.gold.envios_tipo_ruta")

print("Celda de limpieza lista (comandos comentados por seguridad)")

Celda de limpieza lista (comandos comentados por seguridad)
